## 선택 학습 시작

① 맨 위 시작 셀 실행 → ② 문제 풀기 → ③ 맨 아래 종료 셀 실행 후 종료 입력 → ④ Ctrl+S로 노트북 저장 → ⑤ 이 대화에 다 풀었어라고 알려주기.

공통 \_shared 타이머를 사용하고 기록은 이 교재의 .study_timer에만 남아요. KST = 한국 표준시(Korea Standard Time, 케이에스티; UTC = 협정 세계시(Coordinated Universal Time, 유티시)보다 9시간 빠름). 벽시계 시간에는 휴식·닫은 시간도 포함되며 저장·채팅 시각과 달라요. 모두 실행 중에는 종료를 입력하지 마세요. 타이머 없이도 문제를 풀 수 있어요.


In [ ]:
import stat as _stat
from pathlib import Path as _Path


def _load_shared_timer():
    # 교재 ID·날짜·노트북 이름은 제작 때 고정한 값이에요.
    _lesson_id = "day23-robot-integration"
    _lesson_date = "2026-11-11"
    _notebook_name = "day23.ipynb"

    def _safe(path, directory=False):
        if ".." in path.parts:
            raise RuntimeError("상위 폴더 경로는 사용하지 않아요.")
        for item in [*reversed(path.parents), path]:
            info = item.lstat()
            if (
                _stat.S_ISLNK(info.st_mode)
                or getattr(info, "st_file_attributes", 0) & 0x400
            ):
                raise RuntimeError("바로가기·junction·reparse 경로는 사용하지 않아요.")
        if directory and not _stat.S_ISDIR(info.st_mode):
            raise RuntimeError("예상한 폴더가 아니에요.")
        if not directory and not _stat.S_ISREG(info.st_mode):
            raise RuntimeError("예상한 일반 파일이 아니에요.")
        return path

    _cwd = _safe(_Path.cwd(), directory=True)
    _candidates = []
    for _folder in (_cwd, _cwd / _lesson_id):
        if _folder.name == _lesson_id and (_folder / _notebook_name).is_file():
            _safe(_folder, directory=True)
            _candidates.append(_safe(_folder / _notebook_name))
    if len(_candidates) != 1:
        raise RuntimeError("교재 폴더가 없거나 모호해요.")
    _notebook_path = _candidates[0]
    _shared_dir = _safe(_notebook_path.parent.parent / "_shared", directory=True)
    _helper = _safe(_shared_dir / "study_timer.py")
    # 검사한 파일을 메모리에서 로드해 __pycache__를 만들지 않아요.
    _before = _helper.lstat()
    _source = _helper.read_bytes()
    _after = _safe(_helper).lstat()
    _identity = lambda info: (info.st_dev, info.st_ino, info.st_mtime_ns, info.st_size)
    if _identity(_before) != _identity(_after):
        raise RuntimeError("읽는 동안 공통 도구 파일이 바뀌었어요.")
    _module = {"__name__": "physical_ai_shared_timer"}
    exec(compile(_source, str(_helper), "exec"), _module)
    return _module["create_timer"](
        _lesson_id, _lesson_date, _notebook_name, notebook_path=_notebook_path
    )


try:
    _study_timer = _load_shared_timer()
except (OSError, RuntimeError, SyntaxError):
    print(
        "타이머 경로를 확인하지 못했어요. 루트 또는 교재 폴더에서 열고 _shared 폴더를 함께 보관해 주세요. 기록은 바꾸지 않았어요."
    )
else:
    _study_timer.action("start")

# Day23 센서와 목표 판단을 하나의 프로그램으로 연결하기


2026-11-11 · 실제 약 60분 잠정 목표

**Day23 = 23번째 학습 자료**(day, 데이; 물리 단위 없음). Physical AI = 물리 세계를 관측하고 행동하는 인공지능(피지컬 에이아이; 물리 단위 없음). 오늘은 가상 데이터와 규칙만 사용해요.


**완료**

센서 배열·조건·클래스, 모의 이동과 그래프, 2차원 위치·거리·도착 판단을 실제 풀이로 확인했어요.

**오늘 연결**

기존 도구를 묶어 센서 입력부터 명령과 실행 기록까지 한 번에 처리해요. 예고했던 종합 결과물을 여기서 연결해요.

**다음 단계**

Ubuntu/Git 협업, 벡터·행렬·3차원, ROS2 등 미학습 부분은 이후 연결해요. 이 항목들을 완료했다고 표시하지 않으며 별도 로드맵 문서는 바꾸지 않아요.


**오늘 결과물**

본인의 run_demo() 한 호출로 세 가상 실행과 결과 파일을 만드는 작은 프로그램이에요. 센서 클래스·거리 공식·그래프 기초를 다시 만드는 시험은 생략해요.


**60분 배분**

- 진입·필요 개념 9분 → Q1 우선순위 6분 → Q2 처리 연결 15분.
- Q3 실행 묶음 10분 → Q4 디버깅 10분 → Q5 결과 저장 10분.

실질 분량을 늘리되 같은 문법 문제를 반복하지 않아요. 시간은 잠정 목표이고 빨리 끝나면 멈춰도 돼요. 약 8주로 단축할 방향은 검토하되 학습 속도나 공식 과정 수료를 보장하지 않아요.


## 진입 선택과 처리 흐름


**짧은 진입 선택 · 제출 의무 없음**

센서의 원시 눈금과 보정된 거리 중 어느 값을 정지 기준과 비교해야 하는지 설명할 수 있으면 Q1부터 시작하세요. 확신이 서지 않으면 아래 예제와 제공 도구만 읽고 이어가요. preview 답변이 없다는 이유로 숙달했다고 가정하지 않아요.


**쉬운 뜻**

각 도구의 결과를 다음 도구의 알맞은 입력에 넣어요.

**생활 비유**

재료를 받아 씻고 조리한 뒤 접시에 담는 순서와 비슷해요. 조리 전 재료와 완성한 요리를 섞지 않아요.

**실제 용어**

처리 파이프라인(pipeline, 파이프라인; 단위 없음)과 통합(integration, 인티그레이션; 단위 없음)이에요.


![① 관측 → ② 보정 → ③ 목표 확인 → ④ 상태 선택 → ⑤ 명령·상태 갱신과 기록. 다음 반복은 갱신된 상태를 다시 관측해요.](assets/01-pipeline.png)

① 관측 → ② 보정 → ③ 목표 확인 → ④ 상태 선택 → ⑤ 명령·상태 갱신과 기록. 다음 반복은 갱신된 상태를 다시 관측해요.


**입력 이름과 단위**

- packet = 이번 관측 묶음(패킷; 사전 자체는 단위 없음).
- raw = 보정 전 가상 눈금(raw, 로; count, 카운트). 실제 센서 규격이 아닌 오늘 모델의 약속이에요.
- front_m = 앞쪽 거리(front, 프런트; m, meter, 미터).
- stop_m = 정지 판단 기준 거리(stop, 스톱; m). raw와 바로 비교하지 않아요.


In [ ]:
# 별도 완성 예제: 가상 온도 눈금을 바꾸고 선풍기 상태를 골라요.
def fan_demo(raw_count):
    temperature_c = (raw_count - 5) * 0.1  # 온도, 섭씨 도(degree Celsius)
    fan = "ON" if temperature_c >= 2.0 else "OFF"
    # ON/OFF = 켬/끔(온/오프), 라벨이므로 물리 단위 없음
    return {"temperature_c": temperature_c, "fan": fan}


print(fan_demo(25))
# 예제 출력: {'temperature_c': 2.0, 'fan': 'ON'}

## 지도와 데이터의 약속


**그림 읽기**

x, y = 바닥의 가로·세로 위치(엑스·와이; 각각 m). O = 원점(origin, 오리진; 좌표 (0,0)m). +x는 오른쪽, +y는 위쪽이에요.

S = 시작(start, 스타트), G = 목표(goal, 골), W = 같은 높이의 벽 지점(wall, 월). 점 이름 자체는 단위가 없고 각 좌표의 단위는 m예요.


![예제 전용: S(0.4,0.5)m, G(2.4,0.5)m, W(3.2,0.5)m. 초록 화살표는 앞쪽 관측 방향이에요.](assets/02-example-map.png)

예제 전용: S(0.4,0.5)m, G(2.4,0.5)m, W(3.2,0.5)m. 초록 화살표는 앞쪽 관측 방향이에요.


**오늘 모델의 범위와 변수**

- 오른쪽(+x)으로만 이동하고 y 위치는 바뀌지 않아요. 목표도 시작과 같은 높이에 둬요.
- state = 현재 상태(스테이트; 사전 자체는 단위 없음). xy_m은 현재 (x,y) 위치(m), t_ms는 시작 이후 가상 시각(ms)이에요.
- 이 코드의 t = time(타임, 시각), ms = millisecond(밀리세컨드, 밀리초). 1000ms=1s예요.
- dt_s = 한 번의 이동 간격(delta time, 델타 타임, 디티; s, second, 초). 여기의 d는 이 이름의 일부이고 모든 d가 같은 뜻은 아니에요.
- cruise_mps = 진행 명령 속도(cruise, 크루즈; m/s). mps는 meters per second의 줄임말(엠피에스)이에요.


**가상 관측 예제**

그림의 앞쪽 거리는 3.2−0.4=2.8m예요. 제공 관측기는 이를 60count로 바꾸고, 센서 도구는 (60−4)×0.05=2.8m로 복원해요.

offset = 영점 보정 눈금(오프셋; count), scale = 환산 비율(스케일; m/count)이에요. 이 역변환은 실물 센서 모사가 아니라 도구 연결을 확인하기 위한 이상적인 가상 입력이에요.


## 재사용 도구 센서와 목표 계산


![도구 읽기용 예제: S(0.4,0.5)m, G(2.4,0.5)m. 아래 거리 도구는 현재 점과 목표 점을 받아요.](assets/02-example-map.png)

도구 읽기용 예제: S(0.4,0.5)m, G(2.4,0.5)m. 아래 거리 도구는 현재 점과 목표 점을 받아요.


아래는 이미 통과한 역할의 **제공 도구**예요. 그대로 실행하고 같은 계산을 다시 제출하지 않아도 돼요.

registry = 이름별 센서 도구 모음(레지스트리; 단위 없음). remaining_m은 목표까지 직선거리(m), arrived는 허용오차 안인지 나타내는 참/거짓(bool, 불; 단위 없음)이에요.


In [ ]:
import math


class SensorCalibrator:
    def __init__(self, offset, scale):
        self.offset = offset  # 보정 눈금(count)
        self.scale = scale  # 환산 비율(m/count)

    def apply(self, raw):
        return (raw - self.offset) * self.scale  # 결과 단위 m


def apply_named(registry, raw_by_name):
    return {name: registry[name].apply(value) for name, value in raw_by_name.items()}


def goal_report(current_xy, goal_xy, tolerance_m):
    # current_xy/goal_xy: 현재/목표 (x,y) 위치(m)
    # tolerance_m: 도착 허용오차(tolerance, 톨러런스; m)
    dx_m = goal_xy[0] - current_xy[0]  # 목표 쪽 x 변화량(m)
    dy_m = goal_xy[1] - current_xy[1]  # 목표 쪽 y 변화량(m)
    distance_m = math.hypot(dx_m, dy_m)
    return {"remaining_m": distance_m, "arrived": distance_m <= tolerance_m}


registry = {"front": SensorCalibrator(4, 0.05)}

**단위 점검**

새 과제는 위 함수를 다시 만드는 일이 아니에요. apply_named에 raw 사전을, goal_report에 현재 위치·목표 위치·허용오차를 넣어 알맞게 연결하는 일이에요.


## 제공 관측기와 실행 기록


![입력 위치: S(0,1)m, G(2,1)m, 벽 x=3.5m. observe는 현재 위치에서 벽까지의 거리를 계산해요.](assets/03-clear-input.png)

입력 위치: S(0,1)m, G(2,1)m, 벽 x=3.5m. observe는 현재 위치에서 벽까지의 거리를 계산해요.


world = 가상 환경 설정(월드; 사전 자체는 단위 없음). wall_x_m은 벽의 x 위치(m)예요. observe는 현재 상태로부터 새 관측 묶음을 만들어 줘요.


In [ ]:
def observe(state, world):
    x_m, y_m = state["xy_m"]  # 현재 바닥 위치(m)
    front_m = world["wall_x_m"] - x_m  # 이동 전 앞쪽 거리(m)
    raw_front = 4 + front_m / 0.05  # 가상 센서 눈금(count)
    return {"xy_m": (x_m, y_m), "raw": {"front": raw_front}}

settings = 실험 조건(세팅스; 사전 자체는 단위 없음). status = 선택한 상태 라벨(스테이터스; 단위 없음).

- GO = 진행(고), ARRIVED = 도착 판정(어라이브드), STOP_OBSTACLE = 장애물 정지(스톱 옵스터클). 모두 물리 단위가 없는 라벨이에요.
- speed_mps는 실제 측정 속도가 아닌 명령 속도(m/s)예요.
- 기록의 xy_m·front_m·remaining_m은 이동 전 값, next_xy_m은 한 간격 이동 후 위치(m)예요.


In [ ]:
def finish_step(state, front_m, goal_info, status, settings):
    speed_mps = settings["cruise_mps"] if status == "GO" else 0.0
    x_m, y_m = state["xy_m"]
    next_xy = (x_m + speed_mps * settings["dt_s"], y_m)
    next_state = {
        "xy_m": next_xy,
        "t_ms": state["t_ms"] + int(settings["dt_s"] * 1000),
    }
    row = {
        "t_ms": state["t_ms"],
        "xy_m": list(state["xy_m"]),
        "front_m": front_m,
        "remaining_m": goal_info["remaining_m"],
        "status": status,
        "speed_mps": speed_mps,
        "next_xy_m": list(next_xy),
    }
    return next_state, row  # 새 상태와 새 기록

가속·미끄러짐·오차·제동은 생략하고 명령 속도가 즉시 적용된다고 가정해요. 입력 사전을 변경하지 않는 제공 도구예요.


## 확인 문제 Q1 판단 우선순위


### 확인 문제 Q1

**필수 · 약 6분**

**조건**

1. front_m이 stop_m 이하이면 다른 조건보다 먼저 STOP_OBSTACLE을 반환해요.
2. 그렇지 않고 arrived가 True이면 ARRIVED를 반환해요.
3. 둘 다 아니면 GO를 반환해요.

**할 일**

choose_status(front_m, arrived, stop_m)를 구현하세요. 두 조건이 동시에 참일 때 무엇이 우선인지 먼저 예상하고 실행 결과와 대조하세요. 이 가상 프로그램의 약속이며 실제 안전 보장은 아니에요.

**답할 곳**

아래 Q1 코드 셀과 Q1 답안 셀. 반환값은 위 세 문자열 중 하나예요.


In [ ]:
def choose_status(front_m, arrived, stop_m):
    # 새로 작성: 두 조건이 겹칠 때의 우선순위를 반영하세요.
    if front_m <= stop_m:
        return "STOP_OBSTACLE"
    elif arrived == True:
        return "ARRIVED"
    else:
        return "GO"


for front_m, arrived in [(0.1, True), (0.3, True), (0.8, True), (0.8, False)]:
    print(front_m, arrived, choose_status(front_m, arrived, 0.3))

0.1 True STOP_OBSTACLE
0.3 True STOP_OBSTACLE
0.8 True ARRIVED
0.8 False GO


**Q1 답안**

- 동시에 참일 때 예상한 상태: STOP_OBSTACLE
- 실행 결과와 비교: STOP_OBSTACLE
- 조건 순서가 필요한 이유 한 문장: 두 조건이 겹칠 때 도착 여부보다 장애물 정지를 먼저 판단하기 위해


## 확인 문제 Q2 한 번의 처리를 연결하기


![Q2 입력 위치: S(0,1)m, G(2,1)m, 벽 x=3.5m. 그림은 입력만 보여 주며 다음 위치의 답은 표시하지 않아요.](assets/03-clear-input.png)

Q2 입력 위치: S(0,1)m, G(2,1)m, 벽 x=3.5m. 그림은 입력만 보여 주며 다음 위치의 답은 표시하지 않아요.


![관측은 현재 state로 다시 만들고, 보정된 거리와 목표 판단을 합쳐 상태를 고릅니다.](assets/01-pipeline.png)

관측은 현재 state로 다시 만들고, 보정된 거리와 목표 판단을 합쳐 상태를 고릅니다.


### 확인 문제 Q2

**필수 · 약 15분**

**조건**

step_robot(state, world, settings)는 새 상태와 기록 한 개를 (next_state, row)로 반환해요. 입력 사전은 직접 고치지 않아요.

**할 일과 순서**

1. observe(state, world)로 이번 packet을 받아요.
2. packet의 raw 사전을 apply_named에 전달하고 결과의 front 값을 꺼내요.
3. packet의 xy_m, settings의 goal_xy와 tolerance_m을 goal_report에 전달해요.
4. 보정된 앞쪽 거리, 도착 여부, stop_m을 본인의 choose_status에 넣어요.
5. 결과들을 finish_step에 넘겨 반환해요. 인자 순서는 제공 정의를 보고 맞추세요.

**답할 곳**

아래 Q2 코드 셀. Q2 답안 셀에는 원시 눈금과 보정된 거리를 구분해 적어요. 새 보정식이나 거리 함수를 구현하는 문제는 아니에요.


In [ ]:
def step_robot(state, world, settings):
    # 위 1→5 순서대로 기존 도구를 연결하세요.
    packet = observe(state, world)
    print(packet)
    apply = apply_named(registry, packet["raw"])
    print(apply)
    goal = goal_report(packet["xy_m"], settings["goal_xy"], settings["tolerance_m"])
    print(goal)
    status = choose_status(apply["front"], goal["arrived"], settings["stop_m"])
    print(status)
    return finish_step(state, apply["front"], goal, status, settings)


settings = {
    "goal_xy": (2.0, 1.0),
    "tolerance_m": 0.15,
    "stop_m": 0.3,
    "cruise_mps": 0.5,
    "dt_s": 0.5,
}
initial_state = {"xy_m": (0.0, 1.0), "t_ms": 0}
next_state, row = step_robot(initial_state, {"wall_x_m": 3.5}, settings)
print(row)
print(initial_state)  # 새 상태를 만들고 원본을 유지했는지 확인

{'xy_m': (0.0, 1.0), 'raw': {'front': 74.0}}
{'front': 3.5}
{'remaining_m': 2.0, 'arrived': False}
GO
{'t_ms': 0, 'xy_m': [0.0, 1.0], 'front_m': 3.5, 'remaining_m': 2.0, 'status': 'GO', 'speed_mps': 0.5, 'next_xy_m': [0.25, 1.0]}
{'xy_m': (0.0, 1.0), 't_ms': 0}


**Q2 답안**

- stop_m과 비교하도록 연결한 값과 단위: apply["front"]
- packet의 raw와 보정 결과의 차이: 앞쪽 센서의 보정 전 값은 74.0이고, 보정 후 앞쪽 거리는 3.5m이다
- t_ms 기록은 이동 전/후 중 어느 시점인가: 전


## 제공 실행기 제한된 횟수로 실행하기


**쉬운 뜻**

한 번 처리한 결과를 다음 처리의 입력으로 돌려주고, 끝낼 조건이 생기면 멈춰요.

**생활 비유**

한 걸음 뒤에 주변을 다시 살피되, 목적에 도달하거나 길이 막히면 멈추고 정해진 횟수도 넘기지 않아요.

**실제 용어**

실행 단위 episode(에피소드; 단위 없음), 반복 횟수 제한 step limit(스텝 리밋; 횟수 단위 회)이에요. max_steps는 최대 처리 횟수이며 가상 시각 자체가 아니에요.


In [ ]:
def run_episode(step_function, world, settings, max_steps=20):
    state = {"xy_m": (0.0, 1.0), "t_ms": 0}
    rows = []  # 순서대로 모은 실행 기록(list, 리스트; 단위 없음)
    end_reason = "STEP_LIMIT"  # 횟수 제한 종료라는 라벨
    for _ in range(max_steps):
        state, row = step_function(state, world, settings)
        rows.append(row)
        if row["status"] != "GO":
            end_reason = row["status"]
            break
    return {"rows": rows, "end_reason": end_reason, "final_xy_m": list(state["xy_m"])}

step_function에는 함수 이름 step_robot를 전달해요. 괄호를 붙여 미리 호출하지 않아요. 낯설면 함수 이름을 전달하는 줄만 확인하고 실험으로 이어가요.

STEP_LIMIT(스텝 리밋)은 횟수 제한 종료라는 라벨이며 도착이나 장애물 정지를 뜻하지 않아요.


## 확인 문제 Q3 세 조건을 하나의 진입점으로


![A와 C 입력: S(0,1)m, G(2,1)m, 벽 x=3.5m. 초록은 앞쪽 관측 방향입니다.](assets/03-clear-input.png)

A와 C 입력: S(0,1)m, G(2,1)m, 벽 x=3.5m. 초록은 앞쪽 관측 방향입니다.


![B 입력: 시작·목표는 같고 벽 x만 0.9m로 바꿔요.](assets/04-blocked-input.png)

B 입력: 시작·목표는 같고 벽 x만 0.9m로 바꿔요.


### 확인 문제 Q3

**필수 · 약 10분**

**조건**

- A(clear): world의 wall_x_m=3.5, max_steps=20.
- B(blocked): wall_x_m=0.9, max_steps=20.
- C(short): wall_x_m=3.5, max_steps=2.
- 세 경우 모두 Q2 settings를 사용하고 각 run_episode는 시작 상태를 독립적으로 초기화해요.

**할 일**

run_demo()에서 run_episode를 세 번 호출해 clear, blocked, short를 key로 하는 사전을 반환하세요. 각 값은 해당 run_episode의 반환값 전체예요. 제공 출력 코드로 종료 이유·기록 수·마지막 기록을 확인하세요. A와 B에서는 어떤 관측·판정이 종료를 만들었는지, C를 도착으로 보고하면 안 되는 이유를 설명하세요. 도착이 주어진 상태의 남은 거리를 다시 계산해 제출하지 않아요.

**답할 곳**

아래 Q3 코드 셀과 Q3 답안 셀. 전체 기록 출력은 필요할 때만 확인해요.


In [ ]:
def run_demo():
    # 같은 step_robot와 settings, 다른 world/max_steps를 연결
    return {
        "clear": run_episode(
            step_robot,
            {"wall_x_m": 3.5},
            settings,
            max_steps=20,
        ),
        "blocked": run_episode(
            step_robot,
            {"wall_x_m": 0.9},
            settings,
            max_steps=20,
        ),
        "short": run_episode(
            step_robot,
            {"wall_x_m": 3.5},
            settings,
            max_steps=2,
        ),
    }


demo = run_demo()
for case, result in demo.items():
    print("---------")
    print(case, result["end_reason"], len(result["rows"]))
    print(result["rows"][-1])

{'xy_m': (0.0, 1.0), 'raw': {'front': 74.0}}
{'front': 3.5}
{'remaining_m': 2.0, 'arrived': False}
GO
{'xy_m': (0.25, 1.0), 'raw': {'front': 69.0}}
{'front': 3.25}
{'remaining_m': 1.75, 'arrived': False}
GO
{'xy_m': (0.5, 1.0), 'raw': {'front': 64.0}}
{'front': 3.0}
{'remaining_m': 1.5, 'arrived': False}
GO
{'xy_m': (0.75, 1.0), 'raw': {'front': 59.0}}
{'front': 2.75}
{'remaining_m': 1.25, 'arrived': False}
GO
{'xy_m': (1.0, 1.0), 'raw': {'front': 54.0}}
{'front': 2.5}
{'remaining_m': 1.0, 'arrived': False}
GO
{'xy_m': (1.25, 1.0), 'raw': {'front': 49.0}}
{'front': 2.25}
{'remaining_m': 0.75, 'arrived': False}
GO
{'xy_m': (1.5, 1.0), 'raw': {'front': 44.0}}
{'front': 2.0}
{'remaining_m': 0.5, 'arrived': False}
GO
{'xy_m': (1.75, 1.0), 'raw': {'front': 39.0}}
{'front': 1.75}
{'remaining_m': 0.25, 'arrived': False}
GO
{'xy_m': (2.0, 1.0), 'raw': {'front': 34.0}}
{'front': 1.5}
{'remaining_m': 0.0, 'arrived': True}
ARRIVED
{'xy_m': (0.0, 1.0), 'raw': {'front': 22.0}}
{'front': 0.9}
{'rema

**Q3 답안**

- A 종료 이유를 만든 관측/판정 근거: `clear` {'t_ms': 4000, 'xy_m': [2.0, 1.0], 'front_m': 1.5, 'remaining_m': 0.0, 'status': 'ARRIVED', 'speed_mps': 0.0, 'next_xy_m': [2.0, 1.0]} - 목표까지 남은 거리가 0m여서 도착으로 판정됐다

- B 종료 이유를 만든 관측/판정 근거: `block`{'t_ms': 1500, 'xy_m': [0.75, 1.0], 'front_m': 0.15000000000000002, 'remaining_m': 1.25, 'status': 'STOP_OBSTACLE', 'speed_mps': 0.0, 'next_xy_m': [0.75, 1.0]} - 앞쪽 거리가 약 0.15m로 정지 기준 0.3m 이하여서 장애물 정지로 판정됐다.

- C와 A를 같은 성공으로 분류할 수 없는 이유: `short`는 목표에 도착해서 끝난 게 아니라, 정해진 실행 횟수를 다 써서 끝났기 때문


## 확인 문제 Q4 오래된 센서 입력 찾기


![Q4도 B와 같은 시작·목표·벽을 사용합니다. 상태가 이동해도 옛 상태를 관측에 넣으면 문제가 생겨요.](assets/04-blocked-input.png)

Q4도 B와 같은 시작·목표·벽을 사용합니다. 상태가 이동해도 옛 상태를 관측에 넣으면 문제가 생겨요.


### 확인 문제 Q4

**필수 · 약 10분**

**조건**

broken_step에는 매번 최초 위치로 관측하는 오류 한 곳이 있어요. Q1·Q2 함수를 바꾸지 않고 관측 입력 한 곳만 고쳐요. 실제 장비는 연결하지 않아요.

**할 일과 순서**

1. 어떤 값이 새 상태를 반영하지 못할지 예상해요.
2. 수정 전 제공 호출을 실행하고 처음 세 기록의 xy_m과 front_m을 짝지어 비교해요.
3. 한 곳을 고치고 같은 조건으로 다시 실행해요. 실제 벽까지 거리와 보정된 앞쪽 거리의 관계가 회복됐는지 확인해요.

**답할 곳**

아래 Q4 코드·답안 셀. 고친 줄과 수정 전후 기록 비교를 남겨요. 완성 출력은 미리 주지 않아요.


In [ ]:
def broken_step(state, world, settings):
    # packet = observe({"xy_m": (0.0, 1.0), "t_ms": 0}, world)
    packet = observe(state, world)
    values = apply_named(registry, packet["raw"])
    info = goal_report(state["xy_m"], settings["goal_xy"], settings["tolerance_m"])
    status = choose_status(values["front"], info["arrived"], settings["stop_m"])
    return finish_step(state, values["front"], info, status, settings)


bad_run = run_episode(broken_step, {"wall_x_m": 0.9}, settings, max_steps=12)
for row in bad_run["rows"][:3]:
    print(row["xy_m"], row["front_m"])
# 비교를 기록한 뒤 관측 입력 한 곳을 고치고 다시 실행하세요.

{'xy_m': (0.0, 1.0), 't_ms': 0}
{'xy_m': (0.25, 1.0), 't_ms': 500}
{'xy_m': (0.5, 1.0), 't_ms': 1000}
{'xy_m': (0.75, 1.0), 't_ms': 1500}
[0.0, 1.0] 0.9
[0.25, 1.0] 0.65
[0.5, 1.0] 0.4


**Q4 답안**

- 처음 세 기록에서 발견한 관계: front_m이 똑같다
- 고친 한 줄: packet = observe(state, world)
- 수정 후 같은 조건의 비교와 오류 원인: 처음 위치를 계속 관측해서 앞쪽 거리가 고정됐다. 현재 state를 관측하도록 고치니 0.9 → 0.65 → 0.4m로 줄었다.


## 결과 파일을 남기는 제공 도구


**쉬운 뜻**

결과를 다음에도 읽을 수 있는 글자 파일로 남겨요.

**생활 비유**

실험 일지를 적고 다시 펼쳐 읽는 일과 비슷해요.

**실제 용어**

JSON = 사전·목록 같은 데이터를 주고받는 글자 형식(JavaScript Object Notation, 제이슨; 물리 단위 없음). Python 사전 자체와 같은 물건은 아니며 dump는 쓰기, load는 읽기 함수예요.


![실행 사전 → JSON 파일 → 다시 읽은 사전. 개인 타이머 기록과 다른 학습 결과 파일이에요.](assets/05-result-file.png)

실행 사전 → JSON 파일 → 다시 읽은 사전. 개인 타이머 기록과 다른 학습 결과 파일이에요.


run_id = 실험 이름(run identifier, 런 아이디; 단위 없음). 제공 도구는 새 파일만 만들고 이미 있는 결과는 덮어쓰지 않아요. 다음 실행에서는 demo-002처럼 다른 이름을 쓰면 돼요.


In [ ]:
import json
import re
import stat
from pathlib import Path


def result_path(run_id):
    if not re.fullmatch(r"[a-z0-9-]+", run_id):
        raise ValueError("실험 이름은 영문 소문자·숫자·-만 사용해요.")
    cwd = Path.cwd()
    candidates = [
        p
        for p in (cwd, cwd / "day23-robot-integration")
        if p.name == "day23-robot-integration" and (p / "day23.ipynb").is_file()
    ]
    if len(candidates) != 1:
        raise ValueError("루트 또는 Day23 폴더에서 실행해 주세요.")
    folder = candidates[0] / "results"
    for p in [*reversed(folder.parents), folder]:
        if p.exists():
            info = p.lstat()
            if (
                stat.S_ISLNK(info.st_mode)
                or getattr(info, "st_file_attributes", 0) & 0x400
            ):
                raise ValueError("바로가기 경로에는 저장하지 않아요.")
    folder.mkdir(exist_ok=True)
    return folder / (run_id + ".json")

## 확인 문제 Q5 실행과 파일을 대조하기


**제공 저장 함수** — 쓰기 문법 자체를 다시 만드는 과제가 아니에요.


In [ ]:
def save_report(report, run_id):
    path = result_path(run_id)
    # x 모드: 파일이 이미 있으면 실패해 이전 결과를 보존해요.
    with path.open("x", encoding="utf-8") as file:
        json.dump(report, file, ensure_ascii=False, indent=2)
        file.write("\n")
    return path

### 확인 문제 Q5

**필수 · 약 10분**

**조건**

run_demo() 결과는 사전·목록·숫자·문자열·bool로 이루어져요. 제공 기록의 위치는 JSON과 비교할 수 있도록 list로 저장돼요. 실제 타이머 파일은 사용하지 않아요.

**할 일과 순서**

1. 최신 run_demo() 결과를 새 실험 이름으로 저장해요.
2. 같은 path를 읽기 모드 r로 열어 json.load로 다시 읽어요. file 변수에서 읽는 줄만 빈칸에 작성해요.
3. 다시 읽은 결과가 저장 전과 같은지, 세 case의 종료 이유가 Q3 최신 실행과 맞는지 출력해 비교해요.
4. run_demo, step_robot, 결과 파일의 역할을 각각 한 줄로 설명해요.

**답할 곳**

아래 Q5 코드·답안 셀과 results의 결과 파일. 결과 형식은 세 case가 들어 있는 JSON 한 개예요.


In [ ]:
latest_demo = run_demo()
path = save_report(latest_demo, "demo-001")
with path.open("r", encoding="utf-8") as file:
    # TODO: json.load를 사용해 restored에 읽어 넣으세요.
    restored = json.load(file)
print(restored == latest_demo)
for case, result in restored.items():
    print(case, result["end_reason"])
print("결과 파일:", "results/" + path.name)  # 개인 절대경로 출력 안 함

{'xy_m': (0.0, 1.0), 'raw': {'front': 74.0}}
{'front': 3.5}
{'remaining_m': 2.0, 'arrived': False}
GO
{'xy_m': (0.25, 1.0), 'raw': {'front': 69.0}}
{'front': 3.25}
{'remaining_m': 1.75, 'arrived': False}
GO
{'xy_m': (0.5, 1.0), 'raw': {'front': 64.0}}
{'front': 3.0}
{'remaining_m': 1.5, 'arrived': False}
GO
{'xy_m': (0.75, 1.0), 'raw': {'front': 59.0}}
{'front': 2.75}
{'remaining_m': 1.25, 'arrived': False}
GO
{'xy_m': (1.0, 1.0), 'raw': {'front': 54.0}}
{'front': 2.5}
{'remaining_m': 1.0, 'arrived': False}
GO
{'xy_m': (1.25, 1.0), 'raw': {'front': 49.0}}
{'front': 2.25}
{'remaining_m': 0.75, 'arrived': False}
GO
{'xy_m': (1.5, 1.0), 'raw': {'front': 44.0}}
{'front': 2.0}
{'remaining_m': 0.5, 'arrived': False}
GO
{'xy_m': (1.75, 1.0), 'raw': {'front': 39.0}}
{'front': 1.75}
{'remaining_m': 0.25, 'arrived': False}
GO
{'xy_m': (2.0, 1.0), 'raw': {'front': 34.0}}
{'front': 1.5}
{'remaining_m': 0.0, 'arrived': True}
ARRIVED
{'xy_m': (0.0, 1.0), 'raw': {'front': 22.0}}
{'front': 0.9}
{'rema

**Q5 답안**

- 저장 전후 비교와 세 종료 이유 확인:
  - clear ARRIVED
  - blocked STOP_OBSTACLE
  - short STEP_LIMIT
- run_demo의 역할:같은 step_robot와 settings, 다른 world/max_steps를 연결
- step_robot의 역할: 다음 상태를 만드는 것뿐 아니라, 한 번 관측하고 판단한 뒤 다음 상태와 실행 기록을 돌려주는 함수
- 결과 파일을 남긴 이유: 기록보관용


## 마무리와 제출


**필수 산출물**

- Q1~Q5 코드·답안과 저장 출력이 들어 있는 day23.ipynb.
- 세 실행 결과를 담은 results의 JSON 파일 하나.

**성공 기준**

- raw와 m를 섞지 않고 처리 순서를 연결했어요.
- 장애물 우선순위와 독립 초기화, 횟수 제한의 의미가 실행과 설명에서 맞아요.
- 오래된 관측 입력을 고쳐 수정 전후를 설명했어요.
- 새 결과 파일을 다시 읽어 최신 실행과 비교했어요.


**막힐 때만 확인**

- 함수 대신 사전을 호출함: run_episode의 첫 인자가 step_robot인지 확인해요.
- None 또는 key 오류: Q2 return과 제공 key 철자를 확인해요.
- 기록 길이만 같고 내용이 이상함: 관측에 현재 state를 넣었는지 보세요.
- C를 성공으로 착각함: STEP_LIMIT과 ARRIVED를 구별해요.
- 저장 파일이 이미 있음: 기존 파일을 지우지 말고 새 run_id를 쓰세요.


**짧은 회고 · 선택**

실제 걸린 시간과 막힌 부분을 원할 때 알려 주세요. 막힌 곳이 없으면 빈칸이어도 괜찮아요. 60분을 넘으면 여기서 이어가며 새 필수 과제를 쌓지 않아요.

**모형의 한계**

이상적인 거리 관측과 즉시 속도 적용을 가정한 지도 위의 오른쪽 이동 모델이에요. 경로 탐색·충돌 처리·실물 안전 제어·학습된 AI는 아직 없어요. Ubuntu·Git 협업·행렬·3D·ROS2도 이번 결과로 완료 처리하지 않아요.


**자료와 출처**

노트북 하나에 개념·그림·예제·문제·답안·실행 도구가 모두 있어요. PDF는 같은 내용의 읽기용 사본이에요. 정답·해설은 시도 후에만 제공해요. 개인 맞춤 자료이며 KANT 공식 교재가 아니에요.

기준 과정: https://kant-ai-campus.com/ai-physical

JSON 공식 설명: https://docs.python.org/3.12/library/json.html

공식 종일 부트캠프 주차를 개인 기간으로 환산하지 않아요. 2026-11-11 자료 확인. 신규 설치·유료 GPU·실물 장비가 필요 없고 자료 도착은 기상이나 즉시 공부 요청이 아니에요.


## 선택 새로 측정하기

정말 다시 잴 때만 노트북의 아래 셀에서 enabled=False를 True로 바꾸고 초기화를 입력하세요. 기존 기록을 검증된 백업으로 보존한 뒤 시작 셀을 다시 실행해요.


In [ ]:
import stat as _stat
from pathlib import Path as _Path


def _load_shared_timer():
    # 교재 ID·날짜·노트북 이름은 제작 때 고정한 값이에요.
    _lesson_id = "day23-robot-integration"
    _lesson_date = "2026-11-11"
    _notebook_name = "day23.ipynb"

    def _safe(path, directory=False):
        if ".." in path.parts:
            raise RuntimeError("상위 폴더 경로는 사용하지 않아요.")
        for item in [*reversed(path.parents), path]:
            info = item.lstat()
            if (
                _stat.S_ISLNK(info.st_mode)
                or getattr(info, "st_file_attributes", 0) & 0x400
            ):
                raise RuntimeError("바로가기·junction·reparse 경로는 사용하지 않아요.")
        if directory and not _stat.S_ISDIR(info.st_mode):
            raise RuntimeError("예상한 폴더가 아니에요.")
        if not directory and not _stat.S_ISREG(info.st_mode):
            raise RuntimeError("예상한 일반 파일이 아니에요.")
        return path

    _cwd = _safe(_Path.cwd(), directory=True)
    _candidates = []
    for _folder in (_cwd, _cwd / _lesson_id):
        if _folder.name == _lesson_id and (_folder / _notebook_name).is_file():
            _safe(_folder, directory=True)
            _candidates.append(_safe(_folder / _notebook_name))
    if len(_candidates) != 1:
        raise RuntimeError("교재 폴더가 없거나 모호해요.")
    _notebook_path = _candidates[0]
    _shared_dir = _safe(_notebook_path.parent.parent / "_shared", directory=True)
    _helper = _safe(_shared_dir / "study_timer.py")
    # 검사한 파일을 메모리에서 로드해 __pycache__를 만들지 않아요.
    _before = _helper.lstat()
    _source = _helper.read_bytes()
    _after = _safe(_helper).lstat()
    _identity = lambda info: (info.st_dev, info.st_ino, info.st_mtime_ns, info.st_size)
    if _identity(_before) != _identity(_after):
        raise RuntimeError("읽는 동안 공통 도구 파일이 바뀌었어요.")
    _module = {"__name__": "physical_ai_shared_timer"}
    exec(compile(_source, str(_helper), "exec"), _module)
    return _module["create_timer"](
        _lesson_id, _lesson_date, _notebook_name, notebook_path=_notebook_path
    )


try:
    _study_timer = _load_shared_timer()
except (OSError, RuntimeError, SyntaxError):
    print(
        "타이머 경로를 확인하지 못했어요. 루트 또는 교재 폴더에서 열고 _shared 폴더를 함께 보관해 주세요. 기록은 바꾸지 않았어요."
    )
else:
    enabled = False  # 정말 다시 잴 때만 True로 바꾸세요.
    if enabled:
        _study_timer.action("reset", input("초기화하려면 초기화 입력: "), enabled=True)
    else:
        print("초기화는 비활성화되어 있어요.")

## 선택 학습 종료

완료 뒤 아래 셀에서 직접 종료를 입력하세요. 빈 입력은 취소예요. 모두 실행 중에는 종료를 입력하지 마세요. 종료 뒤 Ctrl+S로 저장하고 이 대화에 알려 주세요.


In [ ]:
import stat as _stat
from pathlib import Path as _Path


def _load_shared_timer():
    # 교재 ID·날짜·노트북 이름은 제작 때 고정한 값이에요.
    _lesson_id = "day23-robot-integration"
    _lesson_date = "2026-11-11"
    _notebook_name = "day23.ipynb"

    def _safe(path, directory=False):
        if ".." in path.parts:
            raise RuntimeError("상위 폴더 경로는 사용하지 않아요.")
        for item in [*reversed(path.parents), path]:
            info = item.lstat()
            if (
                _stat.S_ISLNK(info.st_mode)
                or getattr(info, "st_file_attributes", 0) & 0x400
            ):
                raise RuntimeError("바로가기·junction·reparse 경로는 사용하지 않아요.")
        if directory and not _stat.S_ISDIR(info.st_mode):
            raise RuntimeError("예상한 폴더가 아니에요.")
        if not directory and not _stat.S_ISREG(info.st_mode):
            raise RuntimeError("예상한 일반 파일이 아니에요.")
        return path

    _cwd = _safe(_Path.cwd(), directory=True)
    _candidates = []
    for _folder in (_cwd, _cwd / _lesson_id):
        if _folder.name == _lesson_id and (_folder / _notebook_name).is_file():
            _safe(_folder, directory=True)
            _candidates.append(_safe(_folder / _notebook_name))
    if len(_candidates) != 1:
        raise RuntimeError("교재 폴더가 없거나 모호해요.")
    _notebook_path = _candidates[0]
    _shared_dir = _safe(_notebook_path.parent.parent / "_shared", directory=True)
    _helper = _safe(_shared_dir / "study_timer.py")
    # 검사한 파일을 메모리에서 로드해 __pycache__를 만들지 않아요.
    _before = _helper.lstat()
    _source = _helper.read_bytes()
    _after = _safe(_helper).lstat()
    _identity = lambda info: (info.st_dev, info.st_ino, info.st_mtime_ns, info.st_size)
    if _identity(_before) != _identity(_after):
        raise RuntimeError("읽는 동안 공통 도구 파일이 바뀌었어요.")
    _module = {"__name__": "physical_ai_shared_timer"}
    exec(compile(_source, str(_helper), "exec"), _module)
    return _module["create_timer"](
        _lesson_id, _lesson_date, _notebook_name, notebook_path=_notebook_path
    )


try:
    _study_timer = _load_shared_timer()
except (OSError, RuntimeError, SyntaxError):
    print(
        "타이머 경로를 확인하지 못했어요. 루트 또는 교재 폴더에서 열고 _shared 폴더를 함께 보관해 주세요. 기록은 바꾸지 않았어요."
    )
else:
    _study_timer.finish_interactive()